# Build articles_nite JSON for website

For each row in `articles.articles_nite`:
- If `fk_filelist IS NOT NULL` → pull `text_plain` from `articles.filelist.json_unified`.
- If `fk_filelist IS NULL` → pull `full_transcript` (markdown) from `articles.proquestarticles_nite` (matched by `storeid`) and reformat to the same `<p>...</p>\n` style.

Output: a single JSON file with `id_articles_nite`, `fk_filelist`, `storeid`, `source`, and `text` for every row.

In [1]:
import psycopg2
import psycopg2.extras
import pandas as pd
import json
import re
import markdown
from bs4 import BeautifulSoup
from decimal import Decimal
from pathlib import Path

In [2]:
# --- Configuration ---
DB_PARAMS = {
    'dbname':   'thecall',
    'user':     'postgres',
    'password': 'password',
    'host':     'localhost',
    'port':     '5432'
}

TABLE_SCHEMA = 'articles'
OUTPUT_PATH  = Path(r"D:\development\2026_python\2026_thecall\website7\data\articles_nite.json")

In [3]:
# --- extract_plain_text: pull text_plain out of json_unified ---
def extract_plain_text(json_unified):
    if json_unified is None:
        return None

    # psycopg2 returns jsonb as a dict already; text/json columns come back as str.
    if isinstance(json_unified, str):
        try:
            data = json.loads(json_unified)
        except json.JSONDecodeError:
            return None
    else:
        data = json_unified

    text_plain = data.get('text_plain')
    return text_plain if text_plain else None

In [4]:
# --- markdown_to_paragraph_html: convert markdown to the <p>...</p>\n format ---
def markdown_to_paragraph_html(md_text):
    """Render markdown to HTML, then flatten every block element into a
    single <p>...</p> wrapped paragraph, joined by newlines.
    Matches the simple <p>line</p>\n<p>line</p>... shape used in json_unified."""
    if not md_text:
        return None

    html = markdown.markdown(md_text, extensions=['extra'])
    soup = BeautifulSoup(html, 'html.parser')

    # Pick block-level elements that hold actual content. Skip 'blockquote'
    # because its inner <p> tags are already captured.
    block_tags = ['p', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'li', 'pre']

    paragraphs = []
    for elem in soup.find_all(block_tags):
        # Skip <li> whose only child is a nested <ul>/<ol> (keeps real text only).
        if elem.name == 'li' and elem.find(['ul', 'ol']) and not elem.find(string=True, recursive=False):
            continue
        text = elem.get_text()
        text = re.sub(r'\s+', ' ', text).strip()
        if text:
            paragraphs.append(f'<p>{text}</p>')

    return '\n'.join(paragraphs) if paragraphs else None

In [5]:
# --- Query 1: rows WITH fk_filelist (join into filelist) ---
query_filelist = f"""
    SELECT an.id_articles_nite,
           an.fk_filelist,
           an.storeid,
           fl.json_unified
    FROM {TABLE_SCHEMA}.articles_nite an
    LEFT JOIN {TABLE_SCHEMA}.filelist fl ON fl.id = an.fk_filelist
    WHERE an.fk_filelist IS NOT NULL
"""

# --- Query 2: rows WITHOUT fk_filelist (join into proquestarticles_nite) ---
query_proquest = f"""
    SELECT an.id_articles_nite,
           an.storeid,
           pq.full_transcript
    FROM {TABLE_SCHEMA}.articles_nite an
    LEFT JOIN {TABLE_SCHEMA}.proquestarticles_nite pq ON pq.storeid = an.storeid
    WHERE an.fk_filelist IS NULL
"""

conn = psycopg2.connect(**DB_PARAMS)
cur  = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

cur.execute(query_filelist)
filelist_rows = cur.fetchall()
print(f"Fetched {len(filelist_rows)} rows with fk_filelist")

cur.execute(query_proquest)
proquest_rows = cur.fetchall()
print(f"Fetched {len(proquest_rows)} rows without fk_filelist")

cur.close()
conn.close()
print("Connection closed.")

Fetched 125 rows with fk_filelist
Fetched 316 rows without fk_filelist
Connection closed.


In [6]:
# --- Build unified results list ---
results = []
skipped_filelist = 0
skipped_proquest = 0

for row in filelist_rows:
    text = extract_plain_text(row['json_unified'])
    if text is None:
        skipped_filelist += 1
    results.append({
        'id_articles_nite': row['id_articles_nite'],
        'fk_filelist':      row['fk_filelist'],
        'storeid':          row['storeid'],
        'source':           'filelist',
        'text':             text,
    })

for row in proquest_rows:
    text = markdown_to_paragraph_html(row['full_transcript'])
    if text is None:
        skipped_proquest += 1
    results.append({
        'id_articles_nite': row['id_articles_nite'],
        'fk_filelist':      None,
        'storeid':          row['storeid'],
        'source':           'proquest',
        'text':             text,
    })

results.sort(key=lambda r: r['id_articles_nite'])

print(f"Total records: {len(results)}")
print(f"  filelist rows with no text: {skipped_filelist}")
print(f"  proquest rows with no text: {skipped_proquest}")

Total records: 441
  filelist rows with no text: 1
  proquest rows with no text: 0


In [7]:
# --- Preview first 3 of each source ---
df = pd.DataFrame(results)
df['preview'] = df['text'].apply(lambda t: (t[:80] + '...') if isinstance(t, str) and len(t) > 80 else t)
df[['id_articles_nite', 'fk_filelist', 'storeid', 'source', 'preview']].groupby('source').head(3)

,id_articles_nite,fk_filelist,storeid,source,preview
0,1,275409.0,2823180007,filelist,<p>Spt - Monday Niters 12_26</p>\n<p>Monday N...
1,2,NaN,2823180208,proquest,<p>Monday Nite Footballers Predictions</p>\n<p...
2,3,NaN,2823191102,proquest,"<p>Monday Nite Footballers</p>\n<p>By Jim ""Gra..."
3,4,NaN,2823196566,proquest,<p>Monday Nite Footballers Predictions</p>\n<p...
4,5,332839.0,2823204221,filelist,<p>Spt - Monday Niters 9_26</p>\n<p>Monday Nit...
20,21,56972.0,2823234158,filelist,<p>Spt - Monday Niters (Xmas)</p>\n<p>Monday ...


In [8]:
# --- Save to JSON ---
def _json_default(obj):
    """psycopg2 returns NUMERIC columns (e.g. storeid) as Decimal.
    Convert to int for clean JSON output."""
    if isinstance(obj, Decimal):
        return int(obj)
    raise TypeError(f"Object of type {type(obj).__name__} is not JSON serializable")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False, default=_json_default)
print(f"Saved {len(results)} records to {OUTPUT_PATH}")

Saved 441 records to D:\development\2026_python\2026_thecall\website7\data\articles_nite.json
